In [1]:
from agents import Agent, Production, Chat, Toolkit, Prompt, Production
from pydantic import BaseModel

### **Toolkit**

Toolkits are MCP servers that are externally run. Before initializing a Toolkit object, the MCP server needs to be operational.

In [2]:
utils_toolkit = Toolkit(name = 'Utilities', url = 'http://localhost:9001/mcp')


Load started on 04/01/2026 16:57:23
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/01/2026 16:57:23
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/01/2026 16:57:23
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/01/2026 16:57:23
Loading file Agents.Operation.ToolkitUtilities.cls as udl
Compiling class Agents.Operation.ToolkitUtilities
Compiling routine Agents.Operation.ToolkitUtilities.1
Load finished successfully.


### **Chat**

The Chat API can be used to persist conversations. A chat id can be used to construct a history of that Chat from IRIS instead of needing to maintain it manually. This is particularly important when Enterprise licenses for OpenAI have Zero Data Retention enabled and so OpenAI is not authorized to store the conversation on their servers, the Chat API allows for constructing the conversation from history stored in IRIS.

In [3]:
context = Chat(
    name="travel",
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "We are in Washington DC"},
        {"role": "assistant", "content": "Great, what do you want to do in DC?"}
    ]
)
context

Chat(name='travel', messages=3)

In [4]:
context.messages

[{'role': 'system', 'content': 'You are helpful.'},
 {'role': 'user', 'content': 'We are in Washington DC'},
 {'role': 'assistant', 'content': 'Great, what do you want to do in DC?'}]

In [5]:
context == Chat('travel')

True

### **Prompt**

- The Prompt API is a way to manage and version Prompts. 
- Prompts can be built at runtime using parameters. 
- Prompts Prompts versions can be fetched by a selected version. 
- Variables contained in a prompt can be queried using `get_variables()` method.

In [6]:
bond_system = Prompt(name = 'Agent007', text = 'You are {agent_name}. You always stay in character.')
bond_system.build(agent_name='James Bond')

'You are James Bond. You always stay in character.'

In [7]:
bond_system = Prompt(name = 'Agent007', text = 'Your next mission is of utmost importance, you do not have time to talk.')
bond_system

Prompt(name='Agent007', version=2, text='Your next mission is of utmost importance, you do not have time to talk.')

In [8]:
Prompt('Agent007') == bond_system

True

In [9]:
Prompt('Agent007', version=1)

Prompt(name='Agent007', version=1, text='You are {agent_name}. You always stay in character.')

In [10]:
Prompt('Agent007', version=1).get_variables()

['agent_name']

In [11]:
Prompt('Agent007').delete()
try:
    prompt = Prompt("Agent007")
except ValueError as e:
    print(e)

No prompt found for 'Agent007'


### **Agents**

Agents can be defined by a name, a description (not currently used in any way but can be leveraged in the future for expert selection), and an OpenAI model. Optionally, agents can be configured with a default structured output (modifiable at call time) and a set of toolkits the agent should have access to. These tools are advertised to the LLM specific to access the agent has at a Toolkit level (specifying individual tools inside a Toolkit is not currently supported). Agents must be added to a Production before being used.

In [12]:
molly = Agent(name='Molly', model='gpt-5')
Production('TestAgentProduction', [molly]).start()
molly('What are some summer hiking trails around Boston?')


Load started on 04/01/2026 16:57:24
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/01/2026 16:57:24
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/01/2026 16:57:25
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/01/2026 16:57:25
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/01/2026 16:57:25
Loading file Agents.Message.Request.cls as udl
Compiling class Agents.Message.Request
C

'Here are summer-friendly hiking options around Boston:\n- Middlesex Fells Reservation (Medford/Stoneham): Skyline Trail (7–9 mi) and shady loops near Spot Pond. MBTA: Orange Line to Oak Grove + short bus/walk.\n- Blue Hills Reservation (Milton/Quincy): Skyline (9–10 mi) or shorter loops to Great Blue Hill; shade and Houghton’s Pond for a swim.\n- Breakheart Reservation (Saugus): 3–5 mi loops around lakes; mix of paved and rocky; dog-friendly.\n- Lynn Woods Reservation (Lynn): 2–8 mi wooded rocky trails with views from Burrill Hill.\n- Walden Pond State Reservation (Concord): Easy 1.7 mi loop with swimming; arrive early for parking; commuter rail access.\n- Minute Man National Historical Park (Lexington/Concord): Battle Road Trail (~5 mi), mostly flat and shaded with historic sites.\n- World’s End (Hingham – Trustees): 3–5+ mi breezy coastal carriage roads with skyline views; parking reservation often required.\n- Great Brook Farm State Park (Carlisle): 3–10 mi shady farm roads and sin

Agents can be fetched using only their name. Adding any other parameters will be treated as agent creation.

In [13]:
Agent('Molly') == molly

True

In [14]:
class AlexResponse(BaseModel):
    message: str
    reasoning: str

class MollyResponse(BaseModel):
    text: str
    reasoning: str

alex = Agent(name='Alex', 
             description='Test Agent 1', 
             system_prompt=Prompt(name='alex_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit],
             response_format=AlexResponse)

molly = Agent(name='Molly', 
             description='Test Agent 2', 
             system_prompt=Prompt(name='molly_system', text='You are a helpful agent'),
             model='gpt-5',
             toolkits=[utils_toolkit],
             response_format=MollyResponse)


Load started on 04/01/2026 16:57:44
Loading file Agents.Message.AlexResponse.cls as udl
Compiling class Agents.Message.AlexResponse
Compiling table Agents_Message.AlexResponse
Compiling routine Agents.Message.AlexResponse.1
Load finished successfully.

Load started on 04/01/2026 16:57:44
Loading file Agents.Message.ToolRequest.cls as udl
Compiling class Agents.Message.ToolRequest
Compiling table Agents_Message.ToolRequest
Compiling routine Agents.Message.ToolRequest.1
Load finished successfully.

Load started on 04/01/2026 16:57:44
Loading file Agents.Message.ToolResponse.cls as udl
Compiling class Agents.Message.ToolResponse
Compiling table Agents_Message.ToolResponse
Compiling routine Agents.Message.ToolResponse.1
Load finished successfully.

Load started on 04/01/2026 16:57:44
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/01/2026 16:57:45
Loading file Agents.Ope

In [15]:
Production('TestAgentProduction', [molly, alex]).start()


Load started on 04/01/2026 16:57:46
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/01/2026 16:57:46
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/01/2026 16:57:46
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/01/2026 16:57:47
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/01/2026 16:57:47
Loading file Agents.Message.Request.cls as udl
Compiling class Agents.Message.Request
C

In [16]:
molly(message='What is the weather today?', chat=context)

'{"text": "Washington DC weather today: Cloudy, high 26\\u00b0, low 13\\u00b0.", "reasoning": "Used the provided Utilities.weather tool result for Washington DC; no new tool call was needed."}'

In [17]:
molly(message='Recommend some good food spots for lunch', chat='travel')

'{"text": "Here are solid DC lunch picks across styles:\\n\\nSit-down\\n- Old Ebbitt Grill (Downtown): Classic DC near the White House; crab cakes, raw bar.\\n- Le Diplomate (14th St): French bistro vibes; steak frites, salade ni\\u00e7oise. Reserve if possible.\\n- Zaytinya (Penn Quarter): Eastern Med mezze; hummus, crispy Brussels sprouts, lamb kebabs.\\n- Oyamel (Penn Quarter): Mexico City\\u2013style tacos, ceviches, tableside guac.\\n\\nQuick/casual\\n- A Baked Joint (Mt Vernon Triangle): Big sandwiches, biscuits, strong coffee.\\n- Duke\\u2019s Grocery (Dupont/Navy Yard): One of the city\\u2019s best burgers (Proper Burger).\\n- RASA (Penn Quarter/Navy Yard): Customizable Indian bowls; great veg-friendly options.\\n- Shouk (Downtown/NoMa): Plant-based pitas and bowls; shawarma-spiced mushrooms are great.\\n- Astro Doughnuts & Fried Chicken (Downtown): Spicy fried chicken sandwich + doughnuts.\\n- Union Market (NoMa/NE): Food hall with multiple good stalls (Arepa Zone, TaKorean, m

In [18]:
class Restaurant(BaseModel):
    name: str
    cuisine: str

class TasteAtlas(BaseModel):
    restaurants: list[Restaurant]
    reasoning: str

molly(message='What are some places I would like? I tend to like Italian and Asian cuisines', response_format=TasteAtlas, chat='travel')


Load started on 04/01/2026 16:58:28
Loading file Agents.Message.Restaurant.cls as udl
Compiling class Agents.Message.Restaurant
Compiling routine Agents.Message.Restaurant.1
Load finished successfully.

Load started on 04/01/2026 16:58:29
Loading file Agents.Message.TasteAtlas.cls as udl
Compiling class Agents.Message.TasteAtlas
Compiling table Agents_Message.TasteAtlas
Compiling routine Agents.Message.TasteAtlas.1
Load finished successfully.


'{"restaurants": [{"name": "Sfoglina", "cuisine": "Italian"}, {"name": "L\'Ardente", "cuisine": "Italian"}, {"name": "Centrolina", "cuisine": "Italian"}, {"name": "The Red Hen", "cuisine": "Italian"}, {"name": "2Amys", "cuisine": "Italian"}, {"name": "Daikaya Ramen Shop", "cuisine": "Japanese"}, {"name": "Anju", "cuisine": "Korean"}, {"name": "Thip Khao", "cuisine": "Laotian"}, {"name": "Maketto", "cuisine": "Cambodian/Taiwanese"}, {"name": "CHIKO", "cuisine": "Korean/Chinese"}], "reasoning": "Since you\\u2019re in Washington DC and like Italian and Asian cuisines, here are well-regarded local spots spanning casual to sit-down so you can choose by mood."}'

In [19]:
Production('TestAgentProduction').delete()


Load started on 04/01/2026 16:58:53
Loading file Agents.Utils.Common.cls as udl
Compiling class Agents.Utils.Common
Compiling routine Agents.Utils.Common.1
Load finished successfully.

Load started on 04/01/2026 16:58:54
Loading file Agents.Utils.Production.cls as udl
Compiling class Agents.Utils.Production
Compiling routine Agents.Utils.Production.1
Load finished successfully.

Load started on 04/01/2026 16:58:54
Loading file Agents.Message.LLMRequest.cls as udl
Compiling class Agents.Message.LLMRequest
Compiling table Agents_Message.LLMRequest
Compiling routine Agents.Message.LLMRequest.1
Load finished successfully.

Load started on 04/01/2026 16:58:54
Loading file Agents.Message.LLMResponse.cls as udl
Compiling class Agents.Message.LLMResponse
Compiling table Agents_Message.LLMResponse
Compiling routine Agents.Message.LLMResponse.1
Load finished successfully.

Load started on 04/01/2026 16:58:54
Loading file Agents.Message.Request.cls as udl
Compiling class Agents.Message.Request
C